In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

In [4]:
DATA_PATH = Path("../data/raw/creditcard.csv")

In [5]:
# Load data
df = pd.read_csv(DATA_PATH)
 
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Shape:   {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory:  {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nColumns:\n{list(df.columns)}")
 
print("\n--- dtypes ---")
print(df.dtypes.value_counts())
 
print("\n--- first 5 rows ---")
print(df.head())
 
print("\n--- basic stats ---")
print(df.describe().T[['mean', 'std', 'min', 'max']].round(3))
 

DATASET OVERVIEW
Shape:   284,807 rows × 31 columns
Memory:  70.6 MB

Columns:
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

--- dtypes ---
float64    30
int64       1
Name: count, dtype: int64

--- first 5 rows ---
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 

In [6]:
print("\n" + "=" * 60)
print("CLASS DISTRIBUTION")
print("=" * 60)
 
counts = df['Class'].value_counts()
fraud_rate = counts[1] / len(df)
 
print(f"Legitimate transactions: {counts[0]:>8,}  ({100*(1-fraud_rate):.4f}%)")
print(f"Fraudulent transactions: {counts[1]:>8,}  ({100*fraud_rate:.4f}%)")
print(f"\nImbalance ratio: {counts[0]/counts[1]:.0f}:1")
print(f"\n⚠️  A model that predicts EVERY transaction as legitimate gets:")
print(f"   Accuracy = {100*(1-fraud_rate):.3f}%")
print(f"   But it catches ZERO frauds. Accuracy is useless here.")
print(f"\n✓  Metrics that actually matter:")
print(f"   - Precision: of flagged transactions, how many are actually fraud?")
print(f"   - Recall:    of actual frauds, how many did we catch?")
print(f"   - AUC-PR:    area under the precision-recall curve")
print(f"   - F1:        harmonic mean of precision and recall")
print(f"\n   Business framing:")
print(f"   - False Negative (miss a fraud)   → customer loses money → BAD")
print(f"   - False Positive (flag legit txn) → customer inconvenienced → annoying")
print(f"   - The cost of FN >> cost of FP in fraud detection")
print(f"   → We want HIGH RECALL, will tolerate some FP")
 


CLASS DISTRIBUTION
Legitimate transactions:  284,315  (99.8273%)
Fraudulent transactions:      492  (0.1727%)

Imbalance ratio: 578:1

⚠️  A model that predicts EVERY transaction as legitimate gets:
   Accuracy = 99.827%
   But it catches ZERO frauds. Accuracy is useless here.

✓  Metrics that actually matter:
   - Precision: of flagged transactions, how many are actually fraud?
   - Recall:    of actual frauds, how many did we catch?
   - AUC-PR:    area under the precision-recall curve
   - F1:        harmonic mean of precision and recall

   Business framing:
   - False Negative (miss a fraud)   → customer loses money → BAD
   - False Positive (flag legit txn) → customer inconvenienced → annoying
   - The cost of FN >> cost of FP in fraud detection
   → We want HIGH RECALL, will tolerate some FP


In [7]:
 
print("\n" + "=" * 60)
print("FEATURE AUDIT")
print("=" * 60)
 
# V1-V28 are PCA components of anonymized features
# Time: seconds elapsed since first transaction in dataset
# Amount: transaction amount in euros
# Class: 0=legitimate, 1=fraud
 
pca_features = [f'V{i}' for i in range(1, 29)]
raw_features = ['Time', 'Amount']
 
print(f"\nPCA features (V1-V28): {len(pca_features)} columns")
print("  → Already anonymized and PCA-transformed by the dataset creators")
print("  → Unit variance already (roughly) — but let's verify")
 
print(f"\nRaw features: {raw_features}")
print("  → Time and Amount are in original scale — NEED SCALING")
 
print("\n--- Checking if V features are already scaled ---")
v_stats = df[pca_features].agg(['mean', 'std']).T
print(f"  V feature means: min={v_stats['mean'].min():.3f}, "
      f"max={v_stats['mean'].max():.3f}  (should be ≈ 0)")
print(f"  V feature stds:  min={v_stats['std'].min():.3f}, "
      f"max={v_stats['std'].max():.3f}  (should be ≈ 1)")
 
print("\n--- Time feature ---")
print(f"  Range: {df['Time'].min():.0f}s – {df['Time'].max():.0f}s")
print(f"  That's {df['Time'].max()/3600:.1f} hours of transactions")
print(f"  Std: {df['Time'].std():.1f}  → needs scaling")
 
print("\n--- Amount feature ---")
print(f"  Min: €{df['Amount'].min():.2f}")
print(f"  Max: €{df['Amount'].max():.2f}")
print(f"  Mean: €{df['Amount'].mean():.2f}")
print(f"  Median: €{df['Amount'].median():.2f}")
print(f"  Std: {df['Amount'].std():.2f}  → needs scaling, likely needs log transform")
 
print("\n--- Missing values ---")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("  No missing values. Clean dataset.")
else:
    print(missing[missing > 0])
 


FEATURE AUDIT

PCA features (V1-V28): 28 columns
  → Already anonymized and PCA-transformed by the dataset creators
  → Unit variance already (roughly) — but let's verify

Raw features: ['Time', 'Amount']
  → Time and Amount are in original scale — NEED SCALING

--- Checking if V features are already scaled ---
  V feature means: min=-0.000, max=0.000  (should be ≈ 0)
  V feature stds:  min=0.330, max=1.959  (should be ≈ 1)

--- Time feature ---
  Range: 0s – 172792s
  That's 48.0 hours of transactions
  Std: 47488.1  → needs scaling

--- Amount feature ---
  Min: €0.00
  Max: €25691.16
  Mean: €88.35
  Median: €22.00
  Std: 250.12  → needs scaling, likely needs log transform

--- Missing values ---
  No missing values. Clean dataset.


In [8]:

print("\n" + "=" * 60)
print("FRAUD vs LEGIT: FEATURE DIFFERENCES")
print("=" * 60)
 
legit = df[df['Class'] == 0]
fraud = df[df['Class'] == 1]
 
print(f"\n{'Feature':<10} {'Legit Mean':>12} {'Fraud Mean':>12} "
      f"{'Legit Std':>10} {'Fraud Std':>10}")
print("-" * 56)
 
separating_features = []
for col in pca_features + raw_features:
    lm, fm = legit[col].mean(), fraud[col].mean()
    ls, fs = legit[col].std(), fraud[col].std()
    diff = abs(fm - lm)
    # Normalized difference: how many legit standard deviations apart?
    normalized_diff = diff / (ls + 1e-10)
    if normalized_diff > 0.5:
        separating_features.append((col, normalized_diff, lm, fm))
    print(f"{col:<10} {lm:>12.3f} {fm:>12.3f} {ls:>10.3f} {fs:>10.3f}")
 
print(f"\n--- Features with largest mean separation (normalized by legit std) ---")
separating_features.sort(key=lambda x: -x[1])
for col, sep, lm, fm in separating_features[:10]:
    direction = "↑ in fraud" if fm > lm else "↓ in fraud"
    print(f"  {col:<6}  separation={sep:.2f}  legit={lm:.3f}  "
          f"fraud={fm:.3f}  {direction}")
 


FRAUD vs LEGIT: FEATURE DIFFERENCES

Feature      Legit Mean   Fraud Mean  Legit Std  Fraud Std
--------------------------------------------------------
V1                0.008       -4.772      1.930      6.784
V2               -0.006        3.624      1.636      4.291
V3                0.012       -7.033      1.459      7.111
V4               -0.008        4.542      1.399      2.873
V5                0.005       -3.151      1.357      5.372
V6                0.002       -1.398      1.330      1.858
V7                0.010       -5.569      1.179      7.207
V8               -0.001        0.571      1.161      6.798
V9                0.004       -2.581      1.089      2.501
V10               0.010       -5.677      1.044      4.897
V11              -0.007        3.800      1.003      2.679
V12               0.011       -6.259      0.946      4.654
V13               0.000       -0.109      0.995      1.105
V14               0.012       -6.972      0.897      4.279
V15               0.

In [9]:

print("\n" + "=" * 60)
print("AMOUNT ANALYSIS")
print("=" * 60)
 
print(f"\nLegit  — mean: €{legit['Amount'].mean():.2f}, "
      f"median: €{legit['Amount'].median():.2f}, "
      f"max: €{legit['Amount'].max():.2f}")
print(f"Fraud  — mean: €{fraud['Amount'].mean():.2f}, "
      f"median: €{fraud['Amount'].median():.2f}, "
      f"max: €{fraud['Amount'].max():.2f}")
 
# Fraud amounts by percentile
print(f"\nFraud amount percentiles:")
for p in [25, 50, 75, 90, 95, 99]:
    print(f"  {p}th: €{np.percentile(fraud['Amount'], p):.2f}")
 
print("\n  Insight: Fraudsters often make small test transactions first,")
print("  then larger ones. Look at the 25th percentile of fraud amounts.")


AMOUNT ANALYSIS

Legit  — mean: €88.29, median: €22.00, max: €25691.16
Fraud  — mean: €122.21, median: €9.25, max: €2125.87

Fraud amount percentiles:
  25th: €1.00
  50th: €9.25
  75th: €105.89
  90th: €346.75
  95th: €640.90
  99th: €1357.43

  Insight: Fraudsters often make small test transactions first,
  then larger ones. Look at the 25th percentile of fraud amounts.


In [10]:

print("\n" + "=" * 60)
print("TEMPORAL PATTERNS")
print("=" * 60)
 
# Convert time to hours
df['Hour'] = (df['Time'] / 3600).astype(int) % 48  # 2 days of data
 
hourly = df.groupby('Hour')['Class'].agg(['sum', 'count'])
hourly['fraud_rate'] = hourly['sum'] / hourly['count']
 
peak_fraud_hours = hourly['fraud_rate'].nlargest(5)
print(f"\nHours with highest fraud rate:")
for hour, rate in peak_fraud_hours.items():
    print(f"  Hour {hour:02d}: {100*rate:.3f}% fraud rate")
 
print("\n  Insight: Fraud often spikes at off-hours when humans aren't monitoring.")
 


TEMPORAL PATTERNS

Hours with highest fraud rate:
  Hour 26: 2.055% fraud rate
  Hour 28: 1.508% fraud rate
  Hour 02: 1.332% fraud rate
  Hour 03: 0.714% fraud rate
  Hour 07: 0.683% fraud rate

  Insight: Fraud often spikes at off-hours when humans aren't monitoring.


In [11]:

print("\n" + "=" * 60)
print("POINT-BISERIAL CORRELATION WITH CLASS")
print("=" * 60)
 
correlations = df[pca_features + raw_features + ['Class']].corr()['Class'].drop('Class')
correlations = correlations.abs().sort_values(ascending=False)
 
print("\nTop 10 features by |correlation| with Class:")
for feat, corr in correlations.head(10).items():
    direction = "+" if df[feat].corr(df['Class']) > 0 else "-"
    print(f"  {feat:<8}  |r| = {corr:.4f}  ({direction})")
 
print("\nBottom 5 (least correlated — might be droppable):")
for feat, corr in correlations.tail(5).items():
    print(f"  {feat:<8}  |r| = {corr:.4f}")


POINT-BISERIAL CORRELATION WITH CLASS

Top 10 features by |correlation| with Class:
  V17       |r| = 0.3265  (-)
  V14       |r| = 0.3025  (-)
  V12       |r| = 0.2606  (-)
  V10       |r| = 0.2169  (-)
  V16       |r| = 0.1965  (-)
  V3        |r| = 0.1930  (-)
  V7        |r| = 0.1873  (-)
  V11       |r| = 0.1549  (+)
  V4        |r| = 0.1334  (+)
  V18       |r| = 0.1115  (-)

Bottom 5 (least correlated — might be droppable):
  V26       |r| = 0.0045
  V15       |r| = 0.0042
  V25       |r| = 0.0033
  V23       |r| = 0.0027
  V22       |r| = 0.0008


In [12]:

print("\n" + "=" * 60)
print("SVM-SPECIFIC CONCERNS")
print("=" * 60)
 
print(f"""
1. SCALE — CRITICAL
   V features: roughly zero-mean, unit std ✓ (but verify)
   Time: range 0–{df['Time'].max():.0f}. Needs StandardScaler.
   Amount: range 0–€{df['Amount'].max():.0f}, right-skewed. Needs log + scale.
   
   SVMs are distance-based. An unscaled Amount of €25,000 will dominate
   all V features and make the kernel blind to everything else.
 
2. CLASS WEIGHT — IMPORTANT
   {counts[0]/counts[1]:.0f}:1 imbalance means the SVM's soft margin will be
   dominated by legitimate transactions. We must use class_weight.
   
   Effective C becomes:
     C_fraud  = C × class_weight_fraud    (penalize missing fraud harder)
     C_legit  = C × class_weight_legit
   
   sklearn's SVC uses class_weight='balanced' which sets:
     weight_class = n_samples / (n_classes × n_samples_class)
     weight_fraud = {len(df)} / (2 × {counts[1]}) = {len(df)/(2*counts[1]):.1f}
   
3. DATASET SIZE — THE BIG PROBLEM
   {len(df):,} samples × {len(df):,} samples kernel matrix = {len(df)**2 * 8 / 1e9:.0f} GB RAM
   
   Our scratch SMO cannot handle this. Strategy:
   a) Develop and validate on a stratified subsample (~5,000 rows)
   b) Use sklearn's SVC (libsvm) for the full dataset — it has kernel caching
   c) Or use LinearSVC (primal, no kernel matrix) for the full dataset
   
   We'll do all three and compare.
 
4. EVALUATION STRATEGY
   Never use accuracy. Use:
   - AUC-PR (primary): summarizes P-R curve across all thresholds
   - F1 at threshold 0.5 (secondary)  
   - Recall at 95% precision (business metric: "flag only when confident")
   - Recall at 90% precision (looser: "catch more fraud, more false alarms")
""")
 
print("=" * 60)
print("EDA COMPLETE — Key decisions for modeling:")
print("=" * 60)
print("""
  1. Features to engineer:
     - log1p(Amount) → handles skew
     - Hour of day (cyclical) → temporal signal
     - Amount_to_median_ratio → relative size signal
 
  2. Features to drop: Consider V features with |r| < 0.01
 
  3. Scaling: StandardScaler on everything
 
  4. Sampling strategy for development:
     - Stratified sample: 4,000 legit + 394 fraud (all of them)
     - This fits in memory for our KernelSVM
 
  5. class_weight='balanced' or manual {0:1, 1:576}
 
  6. Evaluation: AUC-PR, not AUC-ROC, not accuracy
""")
 


SVM-SPECIFIC CONCERNS

1. SCALE — CRITICAL
   V features: roughly zero-mean, unit std ✓ (but verify)
   Time: range 0–172792. Needs StandardScaler.
   Amount: range 0–€25691, right-skewed. Needs log + scale.

   SVMs are distance-based. An unscaled Amount of €25,000 will dominate
   all V features and make the kernel blind to everything else.

2. CLASS WEIGHT — IMPORTANT
   578:1 imbalance means the SVM's soft margin will be
   dominated by legitimate transactions. We must use class_weight.

   Effective C becomes:
     C_fraud  = C × class_weight_fraud    (penalize missing fraud harder)
     C_legit  = C × class_weight_legit

   sklearn's SVC uses class_weight='balanced' which sets:
     weight_class = n_samples / (n_classes × n_samples_class)
     weight_fraud = 284807 / (2 × 492) = 289.4

3. DATASET SIZE — THE BIG PROBLEM
   284,807 samples × 284,807 samples kernel matrix = 649 GB RAM

   Our scratch SMO cannot handle this. Strategy:
   a) Develop and validate on a stratified subsa